In [21]:
# Examine the predictions file 
from pathlib import Path

import pandas as pd
import numpy as np
import difflib


prediction_path = Path("../data/full_data/ml_interpretation_risk_output/risk_predictions_for_dashboard.csv")

if not prediction_path.exists():
    raise FileNotFoundError(
        f"Prediction file not found: {prediction_path.resolve()}")

predictions = pd.read_csv(
    prediction_path,
    sep=";",
    decimal=",",
    low_memory=False)

print("Shape:", predictions.shape)

print("\nColumns:")
for column in predictions.columns:
    print(f"- {column}: {predictions[column].dtype}")

Shape: (120000, 81)

Columns:
- product_id: int64
- year_month: object
- region_id: int64
- region: object
- target_month: object
- next_month_risk_label: float64
- split: object
- split_row_number: int64
- model_scope: object
- actual_risk_label_num: float64
- actual_risk_label: object
- predicted_risk_label_num: int64
- predicted_risk_label: object
- prob_low_risk: float64
- prob_medium_risk: float64
- high_risk_probability: float64
- high_risk_probability_pct: float64
- risk_score: float64
- risk_score_0_10: float64
- selected_high_risk_threshold: float64
- is_any_risk_alert: int64
- is_high_risk_alert: int64
- is_false_alarm: float64
- is_missed_high_risk: float64
- product_line: object
- product_category: object
- product_name: object
- lifecycle_stage: object
- units_sold: float64
- revenue: float64
- unique_customers: float64
- avg_discount_pct: float64
- market_demand_index: float64
- competitor_pressure_index: float64
- stockout_flag: int64
- backorder_units: float64
- campaig

In [8]:
KEY_COLUMNS = ["product_id", "region_id", "year_month"]

REQUIRED_COLUMNS = [
    "product_id",
    "region_id",
    "year_month",
    "target_month",
    "split",
    "predicted_risk_label",
    "prob_low_risk",
    "prob_medium_risk",
    "high_risk_probability",
    "risk_score",
    "risk_score_0_10",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in predictions.columns
]

assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)

predictions["year_month"] = pd.to_datetime(
    predictions["year_month"]
)

predictions["target_month"] = pd.to_datetime(
    predictions["target_month"]
)

duplicate_count = predictions.duplicated(
    subset=KEY_COLUMNS
).sum()

probability_sum = predictions[
    [
        "prob_low_risk",
        "prob_medium_risk",
        "high_risk_probability",
    ]
].sum(axis=1)

print("Number of rows:", len(predictions))
print("Duplicate keys:", duplicate_count)

print("\nRows by split:")
print(predictions["split"].value_counts(dropna=False))

print("\nObservation period:")
print(
    predictions["year_month"].min(),
    "to",
    predictions["year_month"].max(),
)

print("\nTarget period:")
print(
    predictions["target_month"].min(),
    "to",
    predictions["target_month"].max(),
)

print("\nRisk levels:")
print(
    predictions["predicted_risk_label"]
    .value_counts(dropna=False)
)

print(
    "\nMaximum probability-sum deviation:",
    (probability_sum - 1).abs().max(),
)

assert duplicate_count == 0, (
    "The prediction key is not unique."
)

assert predictions["high_risk_probability"].between(0, 1).all(), (
    "Some high-risk probabilities are outside 0–1."
)

assert predictions["risk_score"].between(0, 100).all(), (
    "Some risk scores are outside 0–100."
)

assert predictions["risk_score_0_10"].between(0, 10).all(), (
    "Some risk scores are outside 0–10."
)

assert (probability_sum - 1).abs().max() < 1e-6, (
    "Class probabilities do not sum to 1."
)

print("\nAll initial prediction validations passed.")

Number of rows: 120000
Duplicate keys: 0

Rows by split:
split
train      82000
test       20000
valid      16000
scoring     2000
Name: count, dtype: int64

Observation period:
2021-01-01 00:00:00 to 2025-12-01 00:00:00

Target period:
2021-02-01 00:00:00 to 2026-01-01 00:00:00

Risk levels:
predicted_risk_label
Low Risk       62782
Medium Risk    46521
High Risk      10697
Name: count, dtype: int64

Maximum probability-sum deviation: 1.5543122344752192e-15

All initial prediction validations passed.


In [9]:
scoring_predictions = predictions.loc[
    predictions["split"].eq("scoring")
].copy()

latest_target_month = scoring_predictions["target_month"].max()

current_predictions = scoring_predictions.loc[
    scoring_predictions["target_month"].eq(latest_target_month)
].copy()

print("Scoring rows:", len(scoring_predictions))
print("Current rows:", len(current_predictions))
print(
    "Observation month:",
    current_predictions["year_month"].unique(),
)
print(
    "Target month:",
    current_predictions["target_month"].unique(),
)

print("\nCurrent predicted risk distribution:")
print(
    current_predictions["predicted_risk_label"]
    .value_counts(dropna=False)
)

print(
    "\nSelected high-risk thresholds:",
    current_predictions["selected_high_risk_threshold"].unique(),
)

print(
    "\nMissing actual labels:",
    current_predictions["actual_risk_label_num"].isna().sum(),
)

Scoring rows: 2000
Current rows: 2000
Observation month: <DatetimeArray>
['2025-12-01 00:00:00']
Length: 1, dtype: datetime64[ns]
Target month: <DatetimeArray>
['2026-01-01 00:00:00']
Length: 1, dtype: datetime64[ns]

Current predicted risk distribution:
predicted_risk_label
Medium Risk    948
Low Risk       929
High Risk      123
Name: count, dtype: int64

Selected high-risk thresholds: [0.43]

Missing actual labels: 2000


In [ ]:
risk_score_matches_probability = np.allclose(
    current_predictions["risk_score"],
    current_predictions["high_risk_probability"] * 100,
)

percentage_matches_probability = np.allclose(
    current_predictions["high_risk_probability_pct"],
    current_predictions["high_risk_probability"] * 100,
)

expected_high_risk_alert = (
    current_predictions["high_risk_probability"]
    >= current_predictions["selected_high_risk_threshold"]
).astype(int)

alert_flag_matches_threshold = (
    expected_high_risk_alert
    == current_predictions["is_high_risk_alert"]
).all()

print(
    "risk_score = probability × 100:",
    risk_score_matches_probability,
)
print(
    "probability_pct = probability × 100:",
    percentage_matches_probability,
)
print(
    "alert flag matches threshold:",
    alert_flag_matches_threshold,
)

assert len(current_predictions) > 0
assert current_predictions["target_month"].nunique() == 1
assert current_predictions["year_month"].nunique() == 1
assert current_predictions["actual_risk_label_num"].isna().all()
assert percentage_matches_probability
assert alert_flag_matches_threshold

print("\nCurrent scoring snapshot validation passed.")

risk_score = probability × 100: False
probability_pct = probability × 100: True
alert flag matches threshold: True

Current scoring snapshot validation passed.


In [13]:
# Latest top 10 product-region combinations by high-risk probability
display_columns = [
    column
    for column in [
        "product_id",
        "product_name",
        "region_id",
        "region",
        "year_month",
        "target_month",
        "predicted_risk_label",
        "high_risk_probability",
        "risk_score",
        "monitoring_priority",
    ]
    if column in current_predictions.columns
]

top_10_product_regions = (
    current_predictions
    .sort_values(
        "high_risk_probability",
        ascending=False,
    )
    .head(10)
    [display_columns]
)

display(top_10_product_regions)

,product_id,product_name,region_id,region,year_month,target_month,predicted_risk_label,high_risk_probability,risk_score,monitoring_priority
118037,1003,Laptop Pro Model 04,7,North America,2025-12-01,2026-01-01,High Risk,0.956797,95.7,1
119588,1158,Legacy Workstation Model 19,8,Benelux,2025-12-01,2026-01-01,High Risk,0.933259,93.3,1
118053,1005,Laptop Pro Model 06,3,Southern Europe,2025-12-01,2026-01-01,High Risk,0.926109,92.6,1
118223,1022,Laptop Standard Model 03,3,Southern Europe,2025-12-01,2026-01-01,High Risk,0.923806,92.4,1
119498,1149,Legacy Workstation Model 10,8,Benelux,2025-12-01,2026-01-01,High Risk,0.921458,92.1,1
118681,1068,Medical Sensor Model 09,10,APAC,2025-12-01,2026-01-01,High Risk,0.918193,91.8,1
118443,1044,Tablet Enterprise Model 05,3,Southern Europe,2025-12-01,2026-01-01,High Risk,0.917047,91.7,1
118511,1051,Tablet Enterprise Model 12,10,APAC,2025-12-01,2026-01-01,High Risk,0.915772,91.6,1
118331,1033,Laptop Standard Model 14,10,APAC,2025-12-01,2026-01-01,High Risk,0.913745,91.4,1
119181,1118,Industrial Scanner Model 19,10,APAC,2025-12-01,2026-01-01,High Risk,0.906087,90.6,1


In [14]:
expected_risk_score = (
    current_predictions["high_risk_probability"] * 100
).round(1)

risk_score_matches_definition = np.allclose(
    current_predictions["risk_score"],
    expected_risk_score,
)

maximum_rounding_difference = (
    current_predictions["risk_score"]
    - current_predictions["high_risk_probability"] * 100
).abs().max()

print(
    "risk_score = round(probability × 100, 1):",
    risk_score_matches_definition,
)

print(
    "Maximum difference caused by rounding:",
    maximum_rounding_difference,
)

assert risk_score_matches_definition
assert maximum_rounding_difference <= 0.0500001

print("\nCorrected risk-score validation passed.")

risk_score = round(probability × 100, 1): True
Maximum difference caused by rounding: 0.049960198969170005

Corrected risk-score validation passed.


In [24]:
# Spaltenvertrag für aktuellen Snapshot

CURRENT_RISK_COLUMN_GROUPS = {
    "identity_and_time": [
        "product_id",
        "product_name",
        "product_line",
        "product_category",
        "lifecycle_stage",
        "region_id",
        "region",
        "year_month",
        "target_month",
        "split",
        "model_scope",
    ],
    "prediction": [
        "predicted_risk_label_num",
        "predicted_risk_label",
        "prob_low_risk",
        "prob_medium_risk",
        "high_risk_probability",
        "risk_score",
        "risk_score_0_10",
        "selected_high_risk_threshold",
        "is_any_risk_alert",
        "is_high_risk_alert",
    ],
    "business_context": [
        "units_sold",
        "revenue",
        "unique_customers",
        "avg_discount_pct",
        "market_demand_index",
        "competitor_pressure_index",
        "stockout_flag",
        "backorder_units",
        "campaign_flag",
        "website_visits",
        "demo_requests",
        "estimated_gross_profit",
        "estimated_gross_margin_pct",
        "product_unit_share_category_region",
        "unit_share_change_pct",
        "risk_factor_under_trend",
        "risk_factor_volatility",
        "risk_factor_sales_drop",
        "risk_factor_customer_drop",
        "supply_pressure_score",
        "customer_count_growth_pct",
        "market_tailwind_score",
        "high_competition_flag",
        "margin_gap_vs_target",
        "campaign_active_with_no_sales",
        "commercial_pressure_score",
        "demand_supply_mismatch_flag",
        "revenue_at_risk_proxy",
    ],
    "operations": [
        "business_risk_indicators",
        "recommended_action",
        "business_indicator_count",
        "monitoring_priority",
    ],
}

In [26]:
current_risk_columns = [
    column
    for group_columns in CURRENT_RISK_COLUMN_GROUPS.values()
    for column in group_columns
]

assert len(current_risk_columns) == len(set(current_risk_columns)), (
    "A column appears in more than one group."
)

missing_selected_columns = sorted(
    set(current_risk_columns) - set(predictions.columns)
)

print("Number of selected columns:", len(current_risk_columns))
print("Missing selected columns:", missing_selected_columns)

for group_name, columns in CURRENT_RISK_COLUMN_GROUPS.items():
    print(f"{group_name}: {len(columns)} columns")

assert not missing_selected_columns

Number of selected columns: 53
Missing selected columns: []
identity_and_time: 11 columns
prediction: 10 columns
business_context: 28 columns
operations: 4 columns


In [27]:
current_risk_snapshot = current_predictions[
    current_risk_columns
].copy()

print("Current snapshot shape:", current_risk_snapshot.shape)

missing_profile = (
    current_risk_snapshot
    .isna()
    .sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
)

print("\nSelected columns with missing values:")
print(
    missing_profile
    if not missing_profile.empty
    else "No missing values"
)

assert len(current_risk_snapshot) == 2_000
assert not current_risk_snapshot.duplicated(
    ["product_id", "region_id", "year_month"]
).any()

print("\nCurrent risk snapshot contract passed.")

Current snapshot shape: (2000, 53)

Selected columns with missing values:
unit_share_change_pct         605
customer_count_growth_pct     605
avg_discount_pct              586
estimated_gross_margin_pct    580
margin_gap_vs_target          580
dtype: int64

Current risk snapshot contract passed.


In [28]:
# Funktionstest mit einer echten Frage

top_10_current_risks = (
    current_risk_snapshot
    .sort_values(
        by="high_risk_probability",
        ascending=False,
    )
    .head(10)
)

result_columns = [
    "product_id",
    "product_name",
    "region_id",
    "region",
    "target_month",
    "predicted_risk_label",
    "high_risk_probability",
    "risk_score",
    "revenue",
    "business_risk_indicators",
]

display(top_10_current_risks[result_columns])

assert len(top_10_current_risks) == 10

assert top_10_current_risks[
    "high_risk_probability"
].is_monotonic_decreasing

assert top_10_current_risks["target_month"].nunique() == 1

print("Top-10 query test passed.")

,product_id,product_name,region_id,region,target_month,predicted_risk_label,high_risk_probability,risk_score,revenue,business_risk_indicators
118037,1003,Laptop Pro Model 04,7,North America,2026-01-01,High Risk,0.956797,95.7,203015.94,Supply pressure
119588,1158,Legacy Workstation Model 19,8,Benelux,2026-01-01,High Risk,0.933259,93.3,110108.69,Supply pressure
118053,1005,Laptop Pro Model 06,3,Southern Europe,2026-01-01,High Risk,0.926109,92.6,595594.74,Supply pressure
118223,1022,Laptop Standard Model 03,3,Southern Europe,2026-01-01,High Risk,0.923806,92.4,92141.64,Supply pressure
119498,1149,Legacy Workstation Model 10,8,Benelux,2026-01-01,High Risk,0.921458,92.1,69489.22,Supply pressure; Margin below target
118681,1068,Medical Sensor Model 09,10,APAC,2026-01-01,High Risk,0.918193,91.8,10748.29,Supply pressure; Margin below target
118443,1044,Tablet Enterprise Model 05,3,Southern Europe,2026-01-01,High Risk,0.917047,91.7,63994.41,Supply pressure
118511,1051,Tablet Enterprise Model 12,10,APAC,2026-01-01,High Risk,0.915772,91.6,35109.01,Supply pressure
118331,1033,Laptop Standard Model 14,10,APAC,2026-01-01,High Risk,0.913745,91.4,28169.59,Supply pressure; Margin below target
119181,1118,Industrial Scanner Model 19,10,APAC,2026-01-01,High Risk,0.906087,90.6,989926.32,Supply pressure


Top-10 query test passed.
